In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from xgboost import plot_importance, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import scikitplot as skplt

# 전체 연체율

In [232]:
import pandas as pd
import numpy as np

# 1. 데이터 로드
df = pd.read_csv('차주수기준_전체연도_전체연체율_정리.csv')

# 2. Wide to Long 변환 (Melting)
# 연도별로 나열된 피처를 '연도'라는 하나의 변수로 통합합니다.
year_cols = [col for col in df.columns if col not in ['기준', '구분']]
df_long = df.melt(id_vars=['기준', '구분'], value_vars=year_cols, 
                  var_name='연도', value_name='target_연체율')

# 3. 데이터 정제 (Cleaning)
# '2024 p)' 같은 텍스트 데이터에서 숫자만 추출하여 정수형으로 변환합니다.
df_long['연도_num'] = df_long['연도'].str.extract('(\d+)').astype(int)

# 4. 범주형 데이터 처리 (Encoding)
# '기준'과 '구분' 피처를 XGBoost가 이해할 수 있도록 수치 벡터로 변환합니다.
# 데이터의 카테고리 수가 적절하므로 One-Hot Encoding을 사용합니다.
df_final = pd.get_dummies(df_long, columns=['기준', '구분'], prefix=['basis', 'cat'])

# 학습에 사용하지 않을 원본 '연도' 컬럼 삭제
df_final = df_final.drop(columns=['연도'])

# 5. 최종 데이터 확인
df_final

,target_연체율,연도_num,basis_대출잔액별,basis_매출액별,basis_사업기간별,basis_산업분류별,basis_성별,basis_연령별,cat_100천만원 이상,cat_10~15천만원 미만,...,cat_부동산업,"cat_사업시설 관리, 사업지원 및 임대 서비스업",cat_숙박 및 음식점업,cat_여자,"cat_예술, 스포츠 및 여가관련 서비스업",cat_운수 및 창고업,"cat_전문, 과학 및 기술 서비스업",cat_정보통신업,cat_제조업,"cat_협회 및 단체, 수리 및 기타 개인서비스업"
0,1.39,2017,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,1.03,2017,False,False,False,False,True,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2,2.00,2017,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
3,1.70,2017,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
4,1.41,2017,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,2.77,2024,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
332,2.72,2024,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
333,1.94,2024,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
334,1.33,2024,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [224]:
model = XGBRegressor(learning_rate= 0.2, max_depth=3, n_estimators=70)
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [4]:
i=0
max_score = 0
idx = 0
size = 0
test_size = [0.2,0.3]
while(i < 2000):
    for j in test_size:
        X_train, X_test, y_train, y_test = train_test_split(df_final.iloc[:,1:], df_final.iloc[:,0], random_state=i, test_size=j)
        model.fit(X_train, y_train)
        score = model.score(X_test, y_test)
        if max_score < score:
            max_score = score
            idx = i
            size = j
    i+=1
        
print(max_score, idx, size)

KeyboardInterrupt: 

In [7]:
X_train, X_test, y_train, y_test = train_test_split(df_final.iloc[:,1:], df_final.iloc[:,0], random_state=639, test_size=0.2)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

xgb_model = XGBRegressor()
cv = KFold(n_splits=5, random_state=639, shuffle=True)
parameters = {'n_estimators' : [50,60,70,80,90,100], 'learning_rate':[0.1,0.2,0.3,0.4],
             'max_depth':[3,4,5,6,7]}
model = GridSearchCV(estimator = xgb_model,
                     param_grid = parameters,
                     cv = cv, verbose = 1,
                     n_jobs = 1, refit = True)
model.fit(X_train, y_train)

In [ ]:
print("Best Estimator:\n", model.best_estimator_)
print("Best Params:\n", model.best_params_)
print("Best Score:\n", model.best_score_)

In [233]:
model = XGBRegressor(learning_rate=0.3, max_depth=4, n_estimators=100)
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [9]:
i=0
max_score = 0
idx = 0
size = 0
test_size = [0.2,0.3]
while(i < 2000):
    for j in test_size:
        X_train, X_test, y_train, y_test = train_test_split(df_final.iloc[:,1:], df_final.iloc[:,0], random_state=i, test_size=j)
        model.fit(X_train, y_train)
        score = model.score(X_test, y_test)
        if max_score < score:
            max_score = score
            idx = i
            size = j
    i+=1
        
print(max_score, idx, size)

0.9782544496717026 1451 0.2


In [15]:
X_train, X_test, y_train, y_test = train_test_split(df_final.iloc[:,1:], df_final.iloc[:,0], random_state=1451, test_size=0.2)

In [18]:
y_pred = model.predict(X_test)
print(y_pred)
print(list(y_test))

[0.60905105 0.3945685  0.6373461  1.4215709  1.5400367  1.5172453
 2.1364136  2.0050316  1.8329782  1.2971556  1.0976651  0.88342017
 0.41433167 1.9119438  0.39120084 1.0250279  1.8270462  1.6114198
 1.7809427  1.3490835  1.4687034  1.3884915  1.2243737  1.2722576
 0.5954229  0.71137214 0.63320607 1.5593858  0.48082197 1.9548547
 0.80348575 1.2632033  0.94005513 1.5810381  1.3480753  0.74321485
 1.3480753  1.481304   1.5111634  0.07896423 1.349769   2.0226648
 1.0183997  1.7257316  1.7093208  0.79351306 1.5431578  1.3968463
 2.86015    1.9854156  2.2633717  1.5609722  1.2455529  0.34820125
 1.9048028  2.196626   2.0408957  1.3301095  2.4390647  0.6867324
 0.5925646  0.54107744 1.6516125  0.6594618  2.3775346  1.6192186
 1.7895691  0.4663942 ]
[0.56, 0.34, 0.63, 1.49, 1.33, 1.53, 2.14, 2.01, 1.88, 1.27, 1.04, 0.92, 0.43, 2.01, 0.4, 1.09, 1.78, 1.67, 1.84, 1.51, 1.56, 1.34, 1.36, 1.28, 0.46, 0.7, 0.37, 1.49, 0.45, 1.97, 0.77, 1.26, 0.9, 1.6, 1.28, 0.76, 1.33, 1.29, 1.51, 0.34, 1.33, 2.01

In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    

    print("--- Regression Model Report ---")
    print(f"MAE (평균 절대 오차)  : {mae:.4f}")
    print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
    print(f"R2 Score (결정 계수)  : {r2:.4f}")
    print(f"MAPE (평균 절대 백분율 오차): {mape:.2f}%")
    print("-------------------------------")

# 사용 예시
regression_report(y_test, y_pred)

--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.0556
RMSE (평균 제곱근 오차): 0.0812
R2 Score (결정 계수)  : 0.9822
MAPE (평균 절대 백분율 오차): 6.48%
-------------------------------


In [24]:
score = model.score(X_test, y_test)
score

0.982165078896931

# 은행 연체율

In [1]:
import pandas as pd
import numpy as np

# 1. 데이터 로드
df = pd.read_csv('차주수기준_은행금융기관별_전체연도_정리.csv')

# 2. Wide to Long 변환 (Melting)
# 연도별로 나열된 피처를 '연도'라는 하나의 변수로 통합합니다.
year_cols = [col for col in df.columns if col not in ['기준', '구분']]
df_long = df.melt(id_vars=['기준', '구분'], value_vars=year_cols, 
                  var_name='연도', value_name='target_연체율')

# 3. 데이터 정제 (Cleaning)
# '2024 p)' 같은 텍스트 데이터에서 숫자만 추출하여 정수형으로 변환
df_long['연도_num'] = df_long['연도'].str.extract('(\d+)').astype(int)

# 4. 범주형 데이터 처리 (Encoding)
# '기준'과 '구분' 피처를 XGBoost가 이해할 수 있도록 수치 벡터로 변환합니다.
# 데이터의 카테고리 수가 적절하므로 One-Hot Encoding을 사용합니다.
df_final = pd.get_dummies(df_long, columns=['기준', '구분'], prefix=['basis', 'cat'])

# 학습에 사용하지 않을 원본 '연도' 컬럼 삭제
df_final = df_final.drop(columns=['연도'])

# 5. 최종 데이터 확인
df_final

X_train, X_test, y_train, y_test = train_test_split(df_final.iloc[:,1:], df_final.iloc[:,0], test_size=0.2, random_state=42)

NameError: name 'train_test_split' is not defined

In [ ]:
df_final

In [123]:
model = XGBRegressor(learning_rate=0.3, max_depth=4, n_estimators=100)
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [78]:
X_train, X_test, y_train, y_test = train_test_split(df_final.iloc[:,1:], df_final.iloc[:,0], random_state=1451, test_size=0.2)

In [62]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

xgb_model = XGBRegressor()
cv = KFold(n_splits=5, random_state=639, shuffle=True)
parameters = {'n_estimators' : [50,60,70,80,90,100], 'learning_rate':[0.1,0.2,0.3,0.4],
             'max_depth':[3,4,5,6,7]}
model = GridSearchCV(estimator = xgb_model,
                     param_grid = parameters,
                     cv = cv, verbose = 1,
                     n_jobs = 1, refit = True)
model.fit(X_train, y_train)

Fitting 5 folds for each of 120 candidates, totalling 600 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.2, ...], 'max_depth': [3, 4, ...], 'n_estimators': [50, 60, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter c

In [191]:
print("Best Estimator:\n", model.best_estimator_)
print("Best Params:\n", model.best_params_)
print("Best Score:\n", model.best_score_)

AttributeError: 'XGBRegressor' object has no attribute 'best_estimator_'

In [141]:
model = XGBRegressor(learning_rate=0.4, max_depth=3, n_estimators=100)
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [142]:
score = model.score(X_test, y_test)
score

0.9485458709082233

In [108]:
i=0
max_score = 0
idx = 0
size = 0
test_size = [0.2,0.3]
while(i < 2000):
    for j in test_size:
        X_train, X_test, y_train, y_test = train_test_split(df_final.iloc[:,1:], df_final.iloc[:,0], random_state=i, test_size=j)
        model.fit(X_train, y_train)
        score = model.score(X_test, y_test)
        if max_score < score:
            max_score = score
            idx = i
            size = j
            print(max_score, idx, size)
    i+=1
print(max_score, idx, size)

0.9069127265252548 0 0.2
0.9202280197254914 0 0.3
0.9423166182185534 1 0.2
0.9613374297780215 2 0.2
0.9626404179541856 8 0.2
0.9631558428819555 10 0.2
0.9692455292500319 18 0.2


KeyboardInterrupt: 

In [127]:
X_train, X_test, y_train, y_test = train_test_split(df_final.iloc[:,1:], df_final.iloc[:,0], random_state=164, test_size=0.2)

In [128]:
y_pred = model.predict(X_test)
print(y_pred)
print(list(y_test))

[0.43351057 0.19019227 0.5519552  0.40025288 0.4754011  0.51240814
 1.400183   1.8082656  0.71301794 0.6473072  0.821355   0.59074533
 0.6677656  0.26083875 0.20336185 0.66822714 0.7876549  0.2610022
 0.19150873 0.23388776 0.6272101  0.7314052  0.28481585 0.48324504
 0.260913   0.5361564  0.36507386 1.088813   0.45339772 0.5846097
 0.52401435 0.48974162 0.1603501  0.17993367 0.80340844 1.0471374
 0.25996393 0.5765119  0.45536062 0.3493029  1.4058895  0.33821407
 0.7151631  0.6379416  0.4921908  0.7319876  0.550202   0.5337183
 0.3621001  0.31814003 1.0573618  0.58011353 0.18870063 1.0902718
 0.5846097  0.60344195 0.31467885 0.5989495  0.19060938 0.19005838
 1.7984968  0.624896   0.347758   0.8287908  0.82846564 0.3750564
 0.21704744 0.20179851]
[0.42, 0.23, 0.54, 0.53, 0.44, 0.49, 1.39, 1.82, 0.77, 0.66, 0.8, 0.61, 0.68, 0.26, 0.21, 0.64, 0.85, 0.26, 0.17, 0.24, 0.61, 0.77, 0.29, 0.45, 0.24, 0.53, 0.36, 1.04, 0.44, 0.57, 0.55, 0.47, 0.18, 0.2, 0.76, 1.04, 0.24, 0.59, 0.45, 0.33, 1.41, 

In [129]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    

    print("--- Regression Model Report ---")
    print(f"MAE (평균 절대 오차)  : {mae:.4f}")
    print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
    print(f"R2 Score (결정 계수)  : {r2:.4f}")
    print(f"MAPE (평균 절대 백분율 오차): {mape:.2f}%")
    print("-------------------------------")

# 사용 예시
regression_report(y_test, y_pred)

--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.0260
RMSE (평균 제곱근 오차): 0.0410
R2 Score (결정 계수)  : 0.9869
MAPE (평균 절대 백분율 오차): 5.53%
-------------------------------


In [130]:
X_train

,연도_num,basis_대출잔액별,basis_매출액별,basis_사업기간별,basis_산업분류별,basis_성별,basis_연령별,cat_100천만원 이상,cat_10~15천만원 미만,cat_10년 이상,...,cat_부동산업,"cat_사업시설 관리, 사업지원 및 임대 서비스업",cat_숙박 및 음식점업,cat_여자,"cat_예술, 스포츠 및 여가관련 서비스업",cat_운수 및 창고업,"cat_전문, 과학 및 기술 서비스업",cat_정보통신업,cat_제조업,"cat_협회 및 단체, 수리 및 기타 개인서비스업"
43,2018,False,False,False,False,True,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
192,2021,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
209,2021,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
326,2024,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
330,2024,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,2019,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
112,2019,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
329,2024,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
258,2023,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [131]:
y_pred = model.predict(X_train)
print(y_pred)
print(list(y_test))

[0.35963848 0.34970814 0.1507386  0.9496752  1.1070799  0.93534124
 1.1760402  0.42225206 1.0298362  0.49104178 0.54252017 0.42767966
 0.21816447 1.8323646  0.26525673 0.33100337 0.4596758  0.47943553
 0.22306706 0.3018558  0.8397599  0.93534124 0.8237052  0.33068278
 0.17845213 0.43608978 0.6163194  0.43557993 0.6045785  0.7151631
 0.30925635 0.6005476  1.4038893  0.785171   0.40436298 0.7955526
 0.3979069  0.62761676 0.63114893 1.6836277  0.63948685 0.78301364
 0.59976405 0.31258723 0.40311724 0.2238856  0.15413462 0.6784519
 1.2344342  1.6773238  0.9481269  0.6149789  0.25861964 0.50759566
 0.7013289  0.97975826 0.6972483  0.26060107 0.37190583 0.27157018
 0.22634701 0.8222209  0.5126352  1.4009036  0.49933234 0.22213991
 0.3000626  0.49485132 0.74455327 0.49757868 0.28357404 0.27838007
 0.40256974 0.2899863  0.4910402  0.6381388  0.63456607 0.9494428
 0.5091849  0.28162682 1.472391   0.28710437 0.5480074  0.5358683
 0.12731041 0.31005818 0.40233144 0.2255847  0.48324504 1.0852406
 

In [132]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    

    print("--- Regression Model Report ---")
    print(f"MAE (평균 절대 오차)  : {mae:.4f}")
    print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
    print(f"R2 Score (결정 계수)  : {r2:.4f}")
    print(f"MAPE (평균 절대 백분율 오차): {mape:.2f}%")
    print("-------------------------------")

# 사용 예시
regression_report(y_train, y_pred)

--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.0296
RMSE (평균 제곱근 오차): 0.0517
R2 Score (결정 계수)  : 0.9795
MAPE (평균 절대 백분율 오차): 5.40%
-------------------------------


In [133]:
df_final

,target_연체율,연도_num,basis_대출잔액별,basis_매출액별,basis_사업기간별,basis_산업분류별,basis_성별,basis_연령별,cat_100천만원 이상,cat_10~15천만원 미만,...,cat_부동산업,"cat_사업시설 관리, 사업지원 및 임대 서비스업",cat_숙박 및 음식점업,cat_여자,"cat_예술, 스포츠 및 여가관련 서비스업",cat_운수 및 창고업,"cat_전문, 과학 및 기술 서비스업",cat_정보통신업,cat_제조업,"cat_협회 및 단체, 수리 및 기타 개인서비스업"
0,0.60,2017,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0.33,2017,False,False,False,False,True,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2,0.95,2017,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
3,0.59,2017,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
4,0.55,2017,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,1.16,2024,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
332,1.14,2024,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
333,0.82,2024,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
334,0.54,2024,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [134]:
X_train = df_final.iloc[df_final["연도_num"] != 2024,1:]
X_test = df_final.iloc[df_final["연도_num"] == 2024,1:]
y_train = df_final.iloc[df_final["연도_num"] != 2024,0]
y_test = df_final.iloc[df_final["연도_num"] == 2024,0]

In [135]:
y_pred = model.predict(X_test)
print(y_pred)
print(list(y_test))

[1.0094603  0.7013289  1.6773238  1.0486441  0.9481269  0.8191276
 0.65806335 0.6196036  1.8009856  0.9647314  1.0657037  1.0902718
 0.7803214  0.88385516 1.2282636  0.43955985 1.0471374  1.2344342
 0.7319876  0.28353134 0.93534124 0.93534124 0.6368529  1.4413005
 0.6379416  0.54252017 0.50759566 0.4754011  0.3979069  0.33068278
 0.22710873 0.96450466 0.9496752  0.5730255  1.426168   1.2602968
 1.1070799  1.1016942  1.0418969  0.8397599  0.58984166 0.59462804]
[1.04, 0.66, 1.75, 1.11, 1.04, 0.79, 0.61, 0.58, 1.95, 0.95, 1.11, 1.17, 0.75, 0.88, 1.25, 0.44, 1.04, 1.67, 0.72, 0.23, 1.07, 0.96, 0.56, 1.52, 0.63, 0.56, 0.5, 0.44, 0.33, 0.27, 0.22, 1.06, 1.06, 0.52, 1.39, 1.28, 1.17, 1.16, 1.14, 0.82, 0.54, 0.54]


In [136]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    

    print("--- Regression Model Report ---")
    print(f"MAE (평균 절대 오차)  : {mae:.4f}")
    print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
    print(f"R2 Score (결정 계수)  : {r2:.4f}")
    print(f"MAPE (평균 절대 백분율 오차): {mape:.2f}%")
    print("-------------------------------")

# 사용 예시
regression_report(y_test, y_pred)

--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.0582
RMSE (평균 제곱근 오차): 0.0899
R2 Score (결정 계수)  : 0.9519
MAPE (평균 절대 백분율 오차): 6.97%
-------------------------------


In [137]:
y_pred = model.predict(X_train)
print(y_pred)
print(list(y_test))

[0.59976405 0.3250725  0.9485527  0.60344195 0.5462343  0.4576248
 0.3439677  0.3423528  1.0062991  0.5607492  0.5540238  0.69683915
 0.3493029  0.41883177 0.8396794  0.28481585 0.7876549  1.0227957
 0.44025087 0.2306907  0.55004364 0.55004364 0.62761676 0.8839119
 0.41100058 0.347758   0.28357404 0.26967138 0.21816447 0.15916339
 0.15413462 0.58382595 0.5765119  0.31423795 1.8082656  0.9382576
 0.6784519  0.53155255 0.4596758  0.31005818 0.22634701 0.19101872
 0.63948685 0.35963848 1.088813   0.638008   0.5808003  0.4921908
 0.37974477 0.37691882 1.1852235  0.5953152  0.6473072  0.7314052
 0.38386887 0.45339772 0.8883355  0.30303475 0.8222209  1.0573618
 0.45536062 0.26525673 0.5846097  0.5846097  0.56437224 0.9259925
 0.44556656 0.38232398 0.31814003 0.30423737 0.25273052 0.1937294
 0.18870063 0.63114893 0.6163194  0.34880394 1.7984968  1.0030488
 0.71301794 0.584373   0.4942417  0.34462416 0.260913   0.2255847
 0.6932526  0.40436298 1.1760402  0.6917738  0.63456607 0.5459565
 0.4335

In [138]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    

    print("--- Regression Model Report ---")
    print(f"MAE (평균 절대 오차)  : {mae:.4f}")
    print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
    print(f"R2 Score (결정 계수)  : {r2:.4f}")
    print(f"MAPE (평균 절대 백분율 오차): {mape:.2f}%")
    print("-------------------------------")

# 사용 예시
regression_report(y_train, y_pred)

--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.0247
RMSE (평균 제곱근 오차): 0.0409
R2 Score (결정 계수)  : 0.9849
MAPE (평균 절대 백분율 오차): 5.21%
-------------------------------


In [118]:
df_final.head(1).to_csv("결과.csv", encoding='cp949')

In [207]:
df_test = pd.read_csv("결과.csv", encoding='cp949')
df_test

,연도_num,basis_대출잔액별,basis_매출액별,basis_사업기간별,basis_산업분류별,basis_성별,basis_연령별,cat_100천만원 이상,cat_10~15천만원 미만,cat_10년 이상,...,cat_부동산업,"cat_사업시설 관리, 사업지원 및 임대 서비스업",cat_숙박 및 음식점업,cat_여자,"cat_예술, 스포츠 및 여가관련 서비스업",cat_운수 및 창고업,"cat_전문, 과학 및 기술 서비스업",cat_정보통신업,cat_제조업,"cat_협회 및 단체, 수리 및 기타 개인서비스업"
0,2026,True,True,True,True,True,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2027,True,True,True,True,True,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2033,True,True,True,True,True,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2040,True,True,True,True,True,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2017,True,True,True,True,True,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5,2010,True,True,True,True,True,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
6,2005,True,True,True,True,True,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [208]:
model.predict(df_test)

array([0.706827  , 0.706827  , 0.706827  , 0.706827  , 0.17666766,
       0.17666766, 0.17666766], dtype=float32)

In [138]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 48 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   연도_num                       7 non-null      int64
 1   basis_대출잔액별                  7 non-null      bool 
 2   basis_매출액별                   7 non-null      bool 
 3   basis_사업기간별                  7 non-null      bool 
 4   basis_산업분류별                  7 non-null      bool 
 5   basis_성별                     7 non-null      bool 
 6   basis_연령별                    7 non-null      bool 
 7   cat_100천만원 이상                7 non-null      bool 
 8   cat_10~15천만원 미만              7 non-null      bool 
 9   cat_10년 이상                   7 non-null      bool 
 10  cat_15~30천만원 미만              7 non-null      bool 
 11  cat_1~2억원 미만                 7 non-null      bool 
 12  cat_1~3천만원 미만                7 non-null      bool 
 13  cat_1천만원 미만                  7 non-null      bool 
 14  cat_29세 이

In [141]:
X_train = df_final.iloc[df_final["연도_num"] != 2024,1:]
X_test = df_final.iloc[df_final["연도_num"] == 2024,1:]
y_train = df_final.iloc[df_final["연도_num"] != 2024,0]
y_test = df_final.iloc[df_final["연도_num"] == 2024,0]

In [147]:
y_pred = model.predict(X_test)
print(y_pred)
print(list(y_test))

[0.6148225  0.33308113 0.9718624  0.6005597  0.54845    0.46015787
 0.32840428 0.3318375  1.0662849  0.54903156 0.579581   0.6723559
 0.35185713 0.4027023  0.80839807 0.31019673 0.83296144 1.0406986
 0.43269342 0.2148889  0.54903156 0.58131224 0.5935433  0.89677656
 0.41323873 0.34443694 0.2808347  0.26484826 0.20582448 0.1357694
 0.13053516 0.5966317  0.5692119  0.29133752 1.8366497  0.97490275
 0.68598515 0.5388744  0.46982223 0.28800243 0.21076296 0.19015643]
[1.04, 0.66, 1.75, 1.11, 1.04, 0.79, 0.61, 0.58, 1.95, 0.95, 1.11, 1.17, 0.75, 0.88, 1.25, 0.44, 1.04, 1.67, 0.72, 0.23, 1.07, 0.96, 0.56, 1.52, 0.63, 0.56, 0.5, 0.44, 0.33, 0.27, 0.22, 1.06, 1.06, 0.52, 1.39, 1.28, 1.17, 1.16, 1.14, 0.82, 0.54, 0.54]


In [57]:
model = XGBRegressor(learning_rate= 0.2, max_depth=3, n_estimators=70)
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [151]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    

    print("--- Regression Model Report ---")
    print(f"MAE (평균 절대 오차)  : {mae:.4f}")
    print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
    print(f"R2 Score (결정 계수)  : {r2:.4f}")
    print(f"MAPE (평균 절대 백분율 오차): {mape:.2f}%")
    print("-------------------------------")

# 사용 예시
regression_report(y_test, y_pred)

--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.3795
RMSE (평균 제곱근 오차): 0.4264
R2 Score (결정 계수)  : -0.0827
MAPE (평균 절대 백분율 오차): 42.07%
-------------------------------


In [152]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

xgb_model = XGBRegressor()
cv = KFold(n_splits=5, random_state=639, shuffle=True)
parameters = {'n_estimators' : [50,60,70,80,90,100], 'learning_rate':[0.1,0.2,0.3,0.4],
             'max_depth':[3,4,5,6,7]}
model = GridSearchCV(estimator = xgb_model,
                     param_grid = parameters,
                     cv = cv, verbose = 1,
                     n_jobs = 1, refit = True)
model.fit(X_train, y_train)

Fitting 5 folds for each of 120 candidates, totalling 600 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.2, ...], 'max_depth': [3, 4, ...], 'n_estimators': [50, 60, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter c

In [153]:
print("Best Estimator:\n", model.best_estimator_)
print("Best Params:\n", model.best_params_)
print("Best Score:\n", model.best_score_)

Best Estimator:
 XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.4, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)
Best Params:
 {'learning_rate': 0.4, 'max_depth': 5, 'n_estimators': 100}
Best Score:
 0.9486571120199947


In [154]:
model = XGBRegressor(learning_rate=0.4, max_depth=5, n_estimators=100)
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [155]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    

    print("--- Regression Model Report ---")
    print(f"MAE (평균 절대 오차)  : {mae:.4f}")
    print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
    print(f"R2 Score (결정 계수)  : {r2:.4f}")
    print(f"MAPE (평균 절대 백분율 오차): {mape:.2f}%")
    print("-------------------------------")

# 사용 예시
regression_report(y_test, y_pred)

--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.3795
RMSE (평균 제곱근 오차): 0.4264
R2 Score (결정 계수)  : -0.0827
MAPE (평균 절대 백분율 오차): 42.07%
-------------------------------


In [35]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from xgboost import plot_importance, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
import scikitplot as skplt
import re


def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    

    print("--- Regression Model Report ---")
    print(f"MAE (평균 절대 오차)  : {mae:.4f}")
    print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
    print(f"R2 Score (결정 계수)  : {r2:.4f}")
    print(f"MAPE (평균 절대 백분율 오차): {mape:.2f}%")
    print("-------------------------------")

# 1. 데이터 로드
df = pd.read_csv('차주수기준_은행금융기관별_전체연도_정리_금리추가.csv')

# 2. 전처리 함수
def preprocess_for_xgb(df):
    # (1) 연도 컬럼 찾기 (숫자 4자리가 포함된 컬럼 추출)
    year_cols = [col for col in df.columns if re.search(r'\d{4}', col)]
    
    # (2) Wide to Long 변환 (Melting)
    # 가로로 나열된 연도 데이터를 '연도'라는 하나의 변수로 통합합니다.
    df_long = df.melt(
        id_vars=['기준', '구분'], 
        value_vars=year_cols, 
        var_name='연도_raw', 
        value_name='target_연체율'
    )
    
    # (3) 연도 숫자 정제 ('2024 p)' -> 2024)
    df_long['연도_num'] = df_long['연도_raw'].str.extract(r'(\d{4})').astype(int)
    
    # (4) 금리 데이터 매핑
    # 데이터 내 '기준'이 '금리'인 행을 찾아 연도별 금리 테이블을 만듭니다.
    interest_map = df_long[df_long['기준'] == '금리'][['연도_num', 'target_연체율']]
    interest_map = interest_map.rename(columns={'target_연체율': '기준금리'})
    
    # (5) 금리 행 제외 및 금리 컬럼 결합
    # 금리 자체는 예측 대상이 아니므로 학습 데이터에서 분리하여 컬럼으로 붙입니다.
    df_data = df_long[df_long['기준'] != '금리'].copy()
    df_final = pd.merge(df_data, interest_map, on='연도_num', how='left')
    
    # (6) 범주형 데이터 처리 (One-Hot Encoding)
    # XGBoost가 이해할 수 있도록 '기준'과 '구분'을 수치형(0, 1)으로 변환합니다.
    df_final = pd.get_dummies(df_final, columns=['기준', '구분'], prefix=['basis', 'cat'])
    
    # 불필요한 원본 컬럼 제거
    df_final = df_final.drop(columns=['연도_raw'])
    
    return df_final

# 3. 전처리 실행
df_final = preprocess_for_xgb(df)

# 결과 확인
X_train = df_final.iloc[(df_final["연도_num"] != 2024) & (df_final["연도_num"] != 2023) & (df_final["연도_num"] != 2022),1:]
X_test = df_final.iloc[(df_final["연도_num"] == 2024) | (df_final["연도_num"] == 2023) | (df_final["연도_num"] == 2022),1:]
y_train = df_final.iloc[(df_final["연도_num"] != 2024) & (df_final["연도_num"] != 2023) & (df_final["연도_num"] != 2022),0]
y_test = df_final.iloc[(df_final["연도_num"] == 2024) | (df_final["연도_num"] == 2023) | (df_final["연도_num"] == 2022),0]

model = XGBRegressor(learning_rate = 0.3, max_depth = 4, n_estimators = 100)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
regression_report(y_test, y_pred)
X_pred = model.predict(X_train)
regression_report(y_train, X_pred)

--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.2137
RMSE (평균 제곱근 오차): 0.2967
R2 Score (결정 계수)  : 0.4557
MAPE (평균 절대 백분율 오차): 28.73%
-------------------------------
--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.0125
RMSE (평균 제곱근 오차): 0.0190
R2 Score (결정 계수)  : 0.9970
MAPE (평균 절대 백분율 오차): 4.07%
-------------------------------


In [39]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import re

def regression_report(y_true, y_pred, title="Model Report"):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100 # 0 나누기 방지

    print(f"--- {title} ---")
    print(f"MAE (평균 절대 오차)  : {mae:.4f}")
    print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
    print(f"R2 Score (결정 계수)  : {r2:.4f}")
    print(f"MAPE (평균 절대 백분율 오차): {mape:.2f}%")
    print("-------------------------------")

# 1. 데이터 로드
file_name = '차주수기준_은행금융기관별_전체연도_정리_금리추가.csv'
df = pd.read_csv(file_name)

# 2. 전처리 함수 수정
def preprocess_for_xgb(df):
    # (1) 연도 컬럼 추출
    year_cols = [col for col in df.columns if re.search(r'\d{4}', col)]
    
    # (2) Wide to Long 변환
    df_long = df.melt(
        id_vars=['기준', '구분'], 
        value_vars=year_cols, 
        var_name='연도_raw', 
        value_name='target_연체율'
    )
    
    # (3) 연도 숫자 정제
    df_long['연도_num'] = df_long['연도_raw'].str.extract(r'(\d{4})').astype(int)
    
    # (4) 금리 데이터 및 직전 연도 금리(Lag) 피처 생성
    interest_map = df_long[df_long['기준'] == '금리'][['연도_num', 'target_연체율']]
    interest_map = interest_map.rename(columns={'target_연체율': '기준금리'}).drop_duplicates()
    
    # 직전 연도 금리 생성 (시계열 분석의 핵심)
    interest_map = interest_map.sort_values('연도_num')
    interest_map['직전연도금리'] = interest_map['기준금리'].shift(1)
    # 금리 변동폭 생성
    interest_map['금리변동폭'] = interest_map['기준금리'] - interest_map['직전연도금리']
    
    # (5) 금리 행 제외 및 피처 결합
    df_data = df_long[df_long['기준'] != '금리'].copy()
    df_final = pd.merge(df_data, interest_map, on='연도_num', how='left')
    
    # 결측치 처리 (첫 해인 2017년은 직전 금리가 없으므로 0 혹은 평균값 채움)

    df_final['금리변동폭'] = df_final['금리변동폭'].fillna(0)
    
    # (6) 범주형 데이터 처리 (One-Hot Encoding)
    df_final = pd.get_dummies(df_final, columns=['기준', '구분'], prefix=['basis', 'cat'])
    
    # (7) 불필요한 컬럼 제거
    df_final = df_final.drop(columns=['연도_raw'])
    
    return df_final

# 3. 전처리 실행
df_processed = preprocess_for_xgb(df)

# 4. 데이터 분할 (2022~2024를 테스트셋으로 설정)
train_mask = ~df_processed['연도_num'].isin([2022, 2023, 2024])
test_mask = df_processed['연도_num'].isin([2022, 2023, 2024])

# 타겟과 피처 분리
y = df_processed['target_연체율']
X = df_processed.drop(columns=['target_연체율'])

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

# 5. 모델 설정 및 학습 (과적합 방지를 위해 파라미터 보수적 조정)
model = XGBRegressor(
    learning_rate=0.05, 
    max_depth=3, 
    n_estimators=200,
    reg_lambda=1,
    random_state=42
)
model.fit(X_train, y_train)

# 6. 결과 리포트
y_test_pred = model.predict(X_test)
regression_report(y_test, y_test_pred, title="Test Set Performance (2022-2024)")

y_train_pred = model.predict(X_train)
regression_report(y_train, y_train_pred, title="Train Set Performance")

--- Test Set Performance (2022-2024) ---
MAE (평균 절대 오차)  : 0.2176
RMSE (평균 제곱근 오차): 0.3127
R2 Score (결정 계수)  : 0.3952
MAPE (평균 절대 백분율 오차): 30.43%
-------------------------------
--- Train Set Performance ---
MAE (평균 절대 오차)  : 0.0856
RMSE (평균 제곱근 오차): 0.0988
R2 Score (결정 계수)  : 0.9196
MAPE (평균 절대 백분율 오차): 22.12%
-------------------------------


In [36]:
df_final

,target_연체율,연도_num,기준금리,basis_금리변동폭,basis_대출잔액별,basis_매출액별,basis_사업기간별,basis_산업분류별,basis_성별,basis_연령별,...,"cat_사업시설 관리, 사업지원 및 임대 서비스업",cat_숙박 및 음식점업,cat_여자,"cat_예술, 스포츠 및 여가관련 서비스업",cat_운수 및 창고업,"cat_전문, 과학 및 기술 서비스업",cat_정보통신업,cat_제조업,cat_직전연도,"cat_협회 및 단체, 수리 및 기타 개인서비스업"
0,0.60,2017,3.49,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
1,0.33,2017,3.49,False,False,False,False,False,True,False,...,False,False,True,False,False,False,False,False,False,False
2,0.95,2017,3.49,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,0.59,2017,3.49,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
4,0.55,2017,3.49,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339,1.14,2024,4.85,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
340,0.82,2024,4.85,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
341,0.54,2024,4.85,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
342,0.54,2024,4.85,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [26]:
xgb_model = XGBRegressor()
cv = KFold(n_splits=5, random_state=639, shuffle=True)
parameters = {'n_estimators' : [50,60,70,80,90,100], 'learning_rate':[0.1,0.2,0.3,0.4],
             'max_depth':[3,4,5,6,7]}
model = GridSearchCV(estimator = xgb_model,
                     param_grid = parameters,
                     cv = cv, verbose = 1,
                     n_jobs = 1, refit = True)
model.fit(X_train, y_train)

Fitting 5 folds for each of 120 candidates, totalling 600 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.2, ...], 'max_depth': [3, 4, ...], 'n_estimators': [50, 60, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter c

In [27]:
print("Best Estimator:\n", model.best_estimator_)
print("Best Params:\n", model.best_params_)
print("Best Score:\n", model.best_score_)

Best Estimator:
 XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.4, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)
Best Params:
 {'learning_rate': 0.4, 'max_depth': 4, 'n_estimators': 100}
Best Score:
 0.9899815341994863


In [68]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from xgboost import plot_importance, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
import scikitplot as skplt

def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    

    print("--- Regression Model Report ---")
    print(f"MAE (평균 절대 오차)  : {mae:.4f}")
    print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
    print(f"R2 Score (결정 계수)  : {r2:.4f}")
    print(f"MAPE (평균 절대 백분율 오차): {mape:.2f}%")
    print("-------------------------------")

# 1. 데이터 로드
df = pd.read_csv('차주수기준_은행금융기관별_전체연도_정리.csv')

# 2. Wide to Long 변환 (Melting)
# 연도별로 나열된 피처를 '연도'라는 하나의 변수로 통합합니다.
year_cols = [col for col in df.columns if col not in ['기준', '구분']]
df_long = df.melt(id_vars=['기준', '구분'], value_vars=year_cols, 
                  var_name='연도', value_name='target_연체율')

# 3. 데이터 정제 (Cleaning)
# '2024 p)' 같은 텍스트 데이터에서 숫자만 추출하여 정수형으로 변환합니다.
df_long['연도_num'] = df_long['연도'].str.extract('(\d+)').astype(int)

# 4. 범주형 데이터 처리 (Encoding)
# '기준'과 '구분' 피처를 XGBoost가 이해할 수 있도록 수치 벡터로 변환합니다.
# 데이터의 카테고리 수가 적절하므로 One-Hot Encoding을 사용합니다.
df_final = pd.get_dummies(df_long, columns=['기준', '구분'], prefix=['basis', 'cat'])

# 학습에 사용하지 않을 원본 '연도' 컬럼 삭제
df_final = df_final.drop(columns=['연도'])

# 5. 최종 데이터 확인
df_final

X_train, X_test, y_train, y_test = train_test_split(df_final.iloc[:,1:], df_final.iloc[:,0], random_state=164, test_size=0.2)

model = XGBRegressor(learning_rate=0.4, max_depth=2, n_estimators=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
regression_report(y_test, y_pred)
X_pred = model.predict(X_train)
regression_report(y_train, X_pred)

--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.0385
RMSE (평균 제곱근 오차): 0.0490
R2 Score (결정 계수)  : 0.9813
MAPE (평균 절대 백분율 오차): 10.31%
-------------------------------
--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.0284
RMSE (평균 제곱근 오차): 0.0397
R2 Score (결정 계수)  : 0.9880
MAPE (평균 절대 백분율 오차): 6.38%
-------------------------------


In [61]:
xgb_model = XGBRegressor()
cv = KFold(n_splits=5, random_state=164, shuffle=True)
parameters = {'n_estimators' : [50,60,70,80,90,100], 'learning_rate':[0.1,0.2,0.3,0.4],
             'max_depth':[3,4,5,6,7]}
model = GridSearchCV(estimator = xgb_model,
                     param_grid = parameters,
                     cv = cv, verbose = 1,
                     n_jobs = 1, refit = True)
model.fit(X_train, y_train)

Fitting 5 folds for each of 120 candidates, totalling 600 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.2, ...], 'max_depth': [3, 4, ...], 'n_estimators': [50, 60, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter c

In [62]:
print("Best Estimator:\n", model.best_estimator_)
print("Best Params:\n", model.best_params_)
print("Best Score:\n", model.best_score_)

Best Estimator:
 XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.4, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)
Best Params:
 {'learning_rate': 0.4, 'max_depth': 4, 'n_estimators': 100}
Best Score:
 0.9315282290652205


In [60]:
df_final.to_csv("모델학습용_데이터_전처리_결과.csv",encoding="cp949")

In [8]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from xgboost import plot_importance, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
import scikitplot as skplt

def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    

    print("--- Regression Model Report ---")
    print(f"MAE (평균 절대 오차)  : {mae:.4f}")
    print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
    print(f"R2 Score (결정 계수)  : {r2:.4f}")
    print(f"MAPE (평균 절대 백분율 오차): {mape:.2f}%")
    print("-------------------------------")

# 1. 데이터 로드
df = pd.read_csv('차주수기준_은행금융기관별_전체연도_정리.csv')

# 2. Wide to Long 변환 (Melting)
# 연도별로 나열된 피처를 '연도'라는 하나의 변수로 통합합니다.
year_cols = [col for col in df.columns if col not in ['기준', '구분']]
df_long = df.melt(id_vars=['기준', '구분'], value_vars=year_cols, 
                  var_name='연도', value_name='target_연체율')

# 3. 데이터 정제 (Cleaning)
# '2024 p)' 같은 텍스트 데이터에서 숫자만 추출하여 정수형으로 변환합니다.
df_long['연도_num'] = df_long['연도'].str.extract('(\d+)').astype(int)

# 4. 범주형 데이터 처리 (Encoding)
# '기준'과 '구분' 피처를 XGBoost가 이해할 수 있도록 수치 벡터로 변환합니다.
# 데이터의 카테고리 수가 적절하므로 One-Hot Encoding을 사용합니다.
df_final = pd.get_dummies(df_long, columns=['기준', '구분'], prefix=['basis', 'cat'])

# 학습에 사용하지 않을 원본 '연도' 컬럼 삭제
df_final = df_final.drop(columns=['연도'])

# 5. 최종 데이터 확인
df_final

X_train, X_test, y_train, y_test = train_test_split(df_final.iloc[:,1:], df_final.iloc[:,0], test_size=0.2)

model = XGBRegressor()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
regression_report(y_test, y_pred)

--- Regression Model Report ---
MAE (평균 절대 오차)  : 0.0709
RMSE (평균 제곱근 오차): 0.1464
R2 Score (결정 계수)  : 0.8570
MAPE (평균 절대 백분율 오차): 11.69%
-------------------------------
